# Олимпиада «Россети», 2 этап, 8—9 класс

## Импорт библиотек. Подготовка

In [ ]:
from typing import Tuple, List, Self
import json
import numpy as np
from scipy.spatial.distance import euclidean as euclidean_distance
from fastcore.basics import store_attr
from fastcore.meta import delegates
from matplotlib import pyplot as plt
from matplotlib.ticker import MultipleLocator
import plotly
from plotly import graph_objects as go
from plotly import express as px
from dash import Dash, Output, dcc, html, callback, Input, State

In [ ]:
# from rich.traceback import install
# from rich.console import Console

# install(
#     console=Console(force_jupyter=True),
#     width=100,
#     code_width=100,
#     word_wrap=True,
#     show_locals=True,
# )

In [3]:
# font_dict=dict(family='Arial',
#                size=26,
#                color='black'
#                )

In [4]:
# fig = go.Figure()

# fig.update_xaxes(
#     range=(0, 15),
#     showgrid=True,
#     gridwidth=1,
#     minor=dict(
#         ticklen=2,
#         showgrid=True,
#         gridcolor="Gray",
#         gridwidth=0.25,
#     )
# )

# fig.update_xaxes(
#     title_text='X-axis',
#     showline=True,
#     showticklabels=True,
#     linecolor='black',
#     linewidth=2.4,
#     ticks='outside',
#     tickfont=font_dict,
#     mirror='allticks',
#     tickwidth=2.4,
#     tickcolor='black',
# )

# fig.update_yaxes(
#     range=(0, 15),
#     showgrid=True,
#     gridwidth=1,
#     minor=dict(
#         ticklen=2,
#         showgrid=True,
#         gridcolor="Gray",
#         gridwidth=0.5
#     )
# )

# fig.update_yaxes(title_text='Y-axis',  # axis label
#                  showline=True,  # add line at x=0
#                  linecolor='black',  # line color
#                  linewidth=2.4, # line size
#                  ticks='outside',  # ticks outside axis
#                  tickfont=font_dict, # tick label font
#                  mirror='allticks',  # add ticks to top/right axes
#                  tickwidth=2.4,  # tick width
#                  tickcolor='black',  # tick color
#                  )

# fig.update_layout(plot_bgcolor='black',
#                   autosize=False,
#                   width=750,
#                   height=750)
# fig.show()

In [5]:
# # 1. Create a figure and axes object
# fig, ax = plt.subplots()

# # 2. Turn the grid lines on
# # By default, this applies to major ticks on both axes
# ax.grid(True)

# # Optional: Set axis limits manually to ensure the grid is visible,
# # as Matplotlib determines limits automatically based on data (which is none here).
# ax.set_xlim(0, 15)
# ax.set_ylim(0, 15)

# # Optional: Add minor grid lines for a denser grid
# ax.minorticks_on() # Enables minor ticks
# ax.grid(which='minor', linestyle='-', linewidth=0.1)

# # 3. Display the plot
# plt.show()

## Объявление констант из условий

In [6]:
# Постоянные
OUTAGES_FREQ_PER_KM = 0.05
LOSS_PER_POWER = 200
LOSS_PER_WORK = 50
PRICE_PER_KM = 500000

| Название переменной   | Название в условии кейса                    |
| :-------------------: | :-----------------------------------------: |
| `OUTAGES_FREQ_PER_KM` | $\lambda = 0.05 \ \text{раз/км}$            |
| `LOSS_PER_POWER`      | $у_{\lambda} = 200 \ \text{руб./кВт}$       |
| `LOSS_PER_WORK`       | $у_D = 50 \ \text{руб./кВт} \cdot \text{ч}$ |
| `PRICE_PER_KM`        | Цена за километр линии                      |

In [7]:
# Расположения и мощности
COORDS = [
    (0.5, 4.5),
    (1, 6),
    (2, 9.5),
    (3, 14),
    (3, 2),
    (3, 6.5),
    (3, 13),
    (3.5, 11.5),
    (4, 4.5),
    (4, 0.5),
    (4, 8),
    (4.5, 14.5),
    (5.5, 11),
    (6, 6),
    (6.5, 10.5),
    (7.5, 13),
    (8, 6.5),
    (8, 15),
    (9, 11.5),
    (10, 1.5),
    (10.5, 6),
    (11.5, 3),
    (11.5, 7),
    (12.5, 0),
    (13, 11),
    (13, 3),
    (13.5, 9.5),
    (14, 6),
    (15, 13),
]


POWERS = [
    300,
    20,
    300,
    150,
    20,
    40,
    300,
    150,
    500,
    200,
    20,
    40,
    500,
    20,
    60,
    40,
    40,
    60,
    200,
    150,
    60,
    200,
    60,
    300,
    500,
    500,
    200,
    150,
    40,
]

In [8]:
print(len(COORDS), len(POWERS))
print(list(zip(COORDS, POWERS, strict=True)))

29 29
[((0.5, 4.5), 300), ((1, 6), 20), ((2, 9.5), 300), ((3, 14), 150), ((3, 2), 20), ((3, 6.5), 40), ((3, 13), 300), ((3.5, 11.5), 150), ((4, 4.5), 500), ((4, 0.5), 200), ((4, 8), 20), ((4.5, 14.5), 40), ((5.5, 11), 500), ((6, 6), 20), ((6.5, 10.5), 60), ((7.5, 13), 40), ((8, 6.5), 40), ((8, 15), 60), ((9, 11.5), 200), ((10, 1.5), 150), ((10.5, 6), 60), ((11.5, 3), 200), ((11.5, 7), 60), ((12.5, 0), 300), ((13, 11), 500), ((13, 3), 500), ((13.5, 9.5), 200), ((14, 6), 150), ((15, 13), 40)]


## Представление элементов сети как классов

### `Node` — потребители (вершины графа)

In [ ]:
class Node:
    def __init__(
        self, coords: Tuple[int, int], connections: List["Connection"] = []
    ) -> None:
        self.connections = connections
        self._coords = coords

    @property
    def coords(self) -> Tuple[int, int]:
        return self._coords

    @coords.setter
    def coords(self, value: Tuple[int, int]) -> None:
        raise TypeError(
            f"Попытка изменить координаты нода, который не является ConsumerNode. Истинный класс нода {type(self)}."
        )

    # def add_connection(self, connection: "Connection") -> None:
    #     self.connections.append(connection)
    #     connection.get_opposite_node(self).connections.append(connection)

In [ ]:
class ConsumerNode(Node):
    @delegates(Node.__init__)
    def __init__(self, power: int, **kwargs) -> None:
        super().__init__(**kwargs)
        store_attr()

In [ ]:
class SolderingNode(Node):
    @delegates(Node.__init__)
    def __init__(self, **kwargs) -> None:
        super().__init__(**kwargs)

In [ ]:
class SubstationNode(Node):
    def __init__(self, coords: Tuple[int, int] = (0, 0)) -> None:
        if 0 not in coords:
            raise ValueError("Координаты подстанции {coords} не содержат 0.")

        super().__init__(coords)

    @Node.coords.setter
    def coords(self, value: Tuple[int, int]) -> None:
        if 0 not in value:
            raise ValueError("Координаты подстанции {coords} не содержат 0.")
        self._coords = value

### `Equipment` — оборудование на соединениях

In [ ]:
class Equipment:
    def __init__(self, price: int) -> None:
        self._price = price

    @property
    def price(self) -> int:
        return self._price

In [ ]:
class ManualSwitch(Equipment):
    def __init__(self) -> None:
        super().__init__(50000)

In [ ]:
class AutoSwitch(Equipment):
    def __init__(self) -> None:
        super().__init__(250000)

In [ ]:
class SubstationSwitch(Equipment):
    def __init__(self) -> None:
        super().__init__(150000)

### `Connection` — линия электропередач (ребро)

In [11]:
class NoConnectionError(Exception):
    @delegates(Exception.__init__)
    def __init__(self, **kwargs) -> None:
        super().__init__(**kwargs)

In [12]:
class Connection:
    def __init__(
        self,
        node1: Node,
        node2: Node,
        node1_equip: Equipment | None = None,
        node2_equip: Equipment | None = None,
    ) -> None:
        store_attr()
        print(type(node1))
        if self not in node1.connections:
            node1.connections.append(self)
        if self not in node2.connections:
            node2.connections.append(self)

    @property
    def length(self) -> float:
        return euclidean_distance(self.node1.coords, self.node2.coords)

    @property
    def cost(self) -> float:
        return (
            (self.length * PRICE_PER_KM)
            + self.node1_equip.price
            + self.node2_equip.price
        )

    def get_opposite_node(self, caller: Node) -> Node:
        if caller is self.node1:
            return self.node2
        if caller is self.node2:
            return self.node1
        else:
            raise NoConnectionError(
                f"ПЕРЕХОД НЕВОЗМОЖЕН. Произведена попытка перейти по ребру, соединяющему {self.node1.coords} и {self.node2.coords}, но сaller {caller.coords} не является вершиной этого ребра."
            )

### `Network` — сеть

In [ ]:
class Network:
    def __init__(self, nodes: List[Node], substation_node: SubstationNode) -> None:
        store_attr()

    @classmethod
    def init_from_default(cls) -> Self:
        nodes = []
        for c, p in zip(COORDS, POWERS, strict=True):
            n = ConsumerNode(power=p, coords=c)
            nodes.append(n)
        return cls(nodes, SubstationNode())

    # @classmethod
    # def init_from_custom(cls, nodes: List[Node]) -> Self:
    #     return cls(nodes)

    @property
    def cost(self) -> int:
        self._cost = 0
        self._seen_nodes = set()
        self._seen_connections = set()

        cur = self.consumer_node

    def _get_():
        pass

    def visualize(self) -> None:
        fig, ax = plt.subplots(figsize=(10, 10))

        ax.grid(True)
        ax.set(xlim=(0, 15), ylim=(0, 15))
        ax.minorticks_on()
        ax.xaxis.set_major_locator(MultipleLocator(1))
        ax.yaxis.set_major_locator(MultipleLocator(1))
        ax.xaxis.set_minor_locator(MultipleLocator(0.5))
        ax.yaxis.set_minor_locator(MultipleLocator(0.5))
        ax.grid(which="minor", linestyle="-", linewidth=0.25)

        x, y = list(zip(*[node.coords for node in self.nodes]))
        ax.scatter(x, y, s=100, c="r")

        plt.show()

    def interactive(self) -> Dash:
        MARKER_SIZE = 10
        # Создание приложения
        self.app = Dash()

        # Создания начального графика (без соединений, только точки)
        df = dict.fromkeys(["x", "y", "power", "id"], value=[])

        for i, node in enumerate(self.nodes):
            x, y = node.coords
            power = node.power
            df["x"].append(x)
            df["y"].append(y)
            df["power"].append(power)
            df["id"].append(i)

        main_scatter = px.scatter(
            data_frame=df, x="x", y="y", custom_data="id", height=720, width=720
        ).update_traces(marker_color="black", marker_size=MARKER_SIZE)

        # Разметка
        self.app.layout = [
            html.Div(
                [
                    # График
                    main_scatter := dcc.Graph(figure=main_scatter),
                    html.Div(
                        [
                            html.Div(
                                [
                                    x_coord_slider := dcc.Slider(
                                        min=0, max=15, step=0.5, value=0
                                    ),
                                    html.Div("x"),
                                ]
                            ),
                            html.Div(
                                [
                                    y_coord_slider := dcc.Slider(
                                        min=0, max=15, step=0.5, value=0
                                    ),
                                    html.Div("y"),
                                ]
                            ),
                        ]
                    ),
                    html.Div(
                        [
                            set_substation_coords_button := dcc.Button(
                                children="Установить координаты подстанции"
                            ),
                            substation_coords_change_result := html.Div(children=""),
                            set_soldering_node_coords_button := dcc.Button(
                                children="Установить координаты паечной точки"
                            ),
                            soldering_node_coords_change_result := html.Div(
                                children=""
                            ),
                        ]
                    ),
                    html.Div(
                        [
                            connect_button := dcc.Button(
                                children="Соединить выбранные точки"
                            ),
                            connect_result := html.Div(children=""),
                        ]
                    ),
                    html.Div(
                        [
                            get_cost_button := dcc.Button(
                                children="Рассчитать затраты"
                            ),
                            cost_output := html.Div(children=""),
                        ]
                    ),
                ]
            )
        ]

        @callback(
            Output(main_scatter, "figure"),
            Output(substation_coords_change_result, "children"),
            Input(set_substation_coords_button, "n_clicks"),
            State(x_coord_slider, "value"),
            State(y_coord_slider, "value"),
            State(main_scatter, "figure"),
            prevent_initial_call=True,
        )
        def set_substation_coords_callback(
            n_clicks: int, x: int, y: int, cur_scatter: dict
        ) -> Tuple[go.Figure, str]:
            cur_scatter = go.Figure(cur_scatter)

            if 0 not in (x, y):
                return (
                    cur_scatter,
                    "Ошибка: одна из координат подстанции должна быть равна 0.",
                )
            self.substation_node.coords = (x, y)
            # TODO 3: снизить особую роль питающей подстанции, реализовать custom_data
            new_scatter = cur_scatter.add_trace(
                px.scatter(x=[x], y=[y], custom_data=[])
                .update_traces(marker_color="red", marker_size=MARKER_SIZE)
                .data[0]
            )
            return new_scatter, f"Успешно: координаты подстанции изменены на {x, y}."

        @callback(
            Output(main_scatter, "figure", allow_duplicate=True),
            Output(soldering_node_coords_change_result, "children"),
            Input(set_soldering_node_coords_button, "n_clicks"),
            State(x_coord_slider, "value"),
            State(y_coord_slider, "value"),
            State(main_scatter, "figure"),
            prevent_initial_call=True,
        )
        def set_soldering_node_coords_callback(
            n_clicks: int, x: int, y: int, cur_scatter: dict
        ) -> Tuple[go.Figure, str]:
            cur_scatter = go.Figure(cur_scatter)

            new_node = SolderingNode(coords=(x, y))
            self.nodes.append(new_node)
            # TODO 2: сделать запись custom_data
            new_scatter = cur_scatter.add_trace(
                px.scatter(x=[x], y=[y])
                .update_traces(marker_color="blue", marker_size=MARKER_SIZE)
                .data[0]
            )
            return (
                new_scatter,
                f"Успешно: добавлена паечная точка с координатами {x, y}.",
            )

        @callback(
            Output(
                component_id=main_scatter,
                component_property="figure",
                allow_duplicate=True,
            ),
            Output(component_id=connect_result, component_property="children"),
            Input(component_id=connect_button, component_property="n_clicks"),
            State(component_id=main_scatter, component_property="figure"),
            State(component_id=main_scatter, component_property="selectedData"),
            prevent_initial_call=True,
        )
        def connect_callback(
            n_clicks: int, cur_scatter: dict, selected_data: str
        ) -> Tuple[go.Figure, str]:
            cur_scatter = go.Figure(cur_scatter)

            if selected_data is None or len(selected_data["points"]) == 0:
                return cur_scatter, "Ошибка: ни одной точки не выбрано."

            # TODO 1: сделать обращение по custom_data, а не по индексам
            nodes_to_connect = [
                self.nodes[i["pointIndex"]] for i in selected_data["points"]
            ]
            print([i["pointIndex"]] for i in selected_data["points"])
            if len(nodes_to_connect) == 2:
                Connection(*nodes_to_connect)

                node1, node2 = nodes_to_connect
                x_to_connect = [node1.coords[0], node2.coords[0]]
                y_to_connect = [node1.coords[1], node2.coords[1]]
                new_scatter = cur_scatter.add_trace(
                    px.scatter(x=x_to_connect, y=y_to_connect)
                    .update_traces(mode="lines")
                    .data[0]
                )
                return new_scatter, "Успешно: точки соеденены."
            else:
                return (
                    cur_scatter,
                    f"Ошибка: можно соединить только 2 точки за раз. Вместо 2 точек выбрано {len(nodes_to_connect)} точек.",
                )

        @callback(
            Output(component_id=cost_output, component_property="children"),
            Input(component_id=get_cost_button, component_property="n_clicks"),
            prevent_initial_call=True,
        )
        def get_cost_callback(n_clicks: int) -> str:
            return self.cost

        self.app.run(debug=True, jupyter_mode="external")


myapp = Network.init_from_default().interactive()

Dash app running on http://127.0.0.1:8050/


[<__main__.ConsumerNode object at 0x0000024DB2C17C50>, <__main__.ConsumerNode object at 0x0000024DB2C17110>]
<class '__main__.ConsumerNode'>
[<__main__.ConsumerNode object at 0x0000024DB2C14650>, <__main__.ConsumerNode object at 0x0000024DB2C17110>]
<class '__main__.ConsumerNode'>
[<__main__.ConsumerNode object at 0x0000024DB2C15070>, <__main__.ConsumerNode object at 0x0000024DB2C17EC0>]
<class '__main__.ConsumerNode'>
[<__main__.ConsumerNode object at 0x0000024DB2C14350>, <__main__.ConsumerNode object at 0x0000024DB2C16630>]
<class '__main__.ConsumerNode'>
